In [ ]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
# write your code here
import sqlite3
import csv

conn = sqlite3.connect('../db.sqlite3')
cursor = conn.cursor()
cursor.execute(
    "SELECT "
    "restaurant_orderitem.id AS orderitem_id, "
    "restaurant_orderitem.quantity AS orderitem_quantity, "

    "restaurant_product.id AS product_id, "
    "restaurant_product.price AS product_price, "
    "restaurant_product.name AS product_name, "

    "restaurant_order.id AS order_id, "
    "restaurant_order.datetime AS order_datetime "
    "FROM restaurant_orderitem "
    "LEFT JOIN restaurant_product "
    "on restaurant_orderitem.product_id = restaurant_product.id "
    "LEFT JOIN restaurant_order "
    "on restaurant_orderitem.order_id = restaurant_order.id"
)
filename = "restaurant_data.csv"
rows = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]
with open(filename, "w", newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(column_names)
    writer.writerows(rows)
conn.close()

df = pd.read_csv('restaurant_data.csv')
df.describe()


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
# write your code here
import matplotlib.pyplot as plt
pop_prod = (
    df.groupby("product_name")["orderitem_quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n({val})"
    return inner

plt.pie(
    pop_prod,
    labels=pop_prod.index.tolist(),
    autopct=autopct_format(pop_prod.values),
    startangle=180,
    textprops={'fontsize': 8}
)
plt.title('Popular Products')
plt.show()


# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
# write your code here
item_price = (
    df.groupby(["product_name"])
    .agg(
        total_quantity=("orderitem_quantity", "sum"),
        total_revenue=(
            "orderitem_quantity",
            lambda q: (
                    q * df.loc[q.index, "product_price"]).sum()
        )
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

def autopct_format(values):
    def inner(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n({val:,.0f})"
    return inner

plt.pie(
    item_price["total_revenue"],
    labels=item_price.index.tolist(),
    autopct=autopct_format(item_price["total_revenue"]),
    startangle=180,
    textprops={'fontsize': 8}
)
plt.title('Popular Items')
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
# write your code here
df["Order_item_price"] = df["product_price"] * df["orderitem_quantity"]

df['Hour'] = pd.to_datetime(df.order_datetime, format='%Y-%m-%d %H:%M:%S').dt.hour

pop_hour = (
    df.groupby("Hour")["Order_item_price"]
    .sum()
)

plt.bar(pop_hour.index.to_list(), pop_hour.values)
plt.xticks(pop_hour.index.to_list())
plt.xlabel('Hours')
plt.ylabel('Profit')
plt.title('Popular order hours')
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
# write your code here
df['dayofweek'] = pd.to_datetime(df.order_datetime, format='%Y-%m-%d %H:%M:%S').dt.dayofweek

pop_day = (
    df.groupby("dayofweek")["Order_item_price"]
    .sum()
)

days_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

plt.bar(days_names, pop_day.values)
plt.xlabel('Days')
plt.ylabel('Profit')
plt.title('Popular order days')
plt.xticks(rotation=45)
plt.show()